## Imports and model

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, set_seed
import gc
import random
import numpy as np
import time

model_id = "unsloth/Llama-3.2-1B-Instruct"
query = "why is 42 a special number?"

messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": query},
]

/kaggle/working/inference/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading the model using device = "auto"

In [2]:
# from transformers import pipeline
pipe = pipeline(
    "text-generation",
    model=model_id,
    dtype=torch.float16,
    device_map="auto",
)

outputs = pipe(
    messages, max_new_tokens=200
)
print(outputs[0]["generated_text"][-1]['content'])

Loading weights: 100%|██████████| 146/146 [00:01<00:00, 82.81it/s] 
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to for

42 is indeed a special number, and its uniqueness has been debated among mathematicians and enthusiasts for a long time. Here are some reasons why 42 is considered special:

1. **Fermat's Last Theorem**: In 1637, the famous mathematician Pierre de Fermat proved that there are no integer solutions to the equation a^n + b^n = c^n for n > 2. This theorem was known as Fermat's Last Theorem. The equation a^n + b^n = c^n has been unsolved for centuries, and 42 is the smallest value of n for which this theorem was proved. It's a testament to the power of mathematics and the importance of solving such a complex problem.
2. **The "42 Club"**: In 1987, a group of mathematicians, including Larry Wall (the creator of Perl), claimed to have found a proof of Fermat's Last Theorem. They called themselves the "42 Club" because they believed


Looking at nvidia-smi, once the model is loaded, VRAM on both GPUs is occupied.

In [3]:
!nvidia-smi

Wed Sep  9 19:02:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P0             30W /   70W |    1359MiB /  15360MiB |     12%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
print(pipe.model.hf_device_map)

{'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.norm': 1, 'model.rotary_emb': 1}


The above step verifies that some layers of model exist on GPU 0 whereas the rest are loaded onto GPU 1.

This indicates that when loaded with device set to "auto", the library shards the model and loads it into two GPUs. This is knows as pipeline parallelism, and the activations/calculations after running on first GPU needs to be sent to second GPU to complete the inference, This communication overhead adds delay. We try to measure it in subsequent code.

In [5]:
N = 10

times = []
outputs = pipe(
        messages, max_new_tokens=200
    )

for _ in range(N):
    start = time.perf_counter()
    outputs = pipe(
        messages, max_new_tokens=200
    )
    time_taken = time.perf_counter() - start
    times.append(time_taken)
    print(f"Run {len(times)}: Time taken - {time_taken:.3f} sec")

avg_auto = sum(times) / len(times)
print(f"\nAverage generation time over {N} runs: {avg_auto:.3f}s")

print(outputs[0]["generated_text"][-1]['content'])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 1: Time taken - 6.252 sec


[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 2: Time taken - 6.298 sec


[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Run 7: Time taken - 6.327 sec


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 8: Time taken - 6.728 sec


[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 9: Time taken - 6.999 sec
Run 10: Time taken - 6.742 sec

Average generation time over 10 runs: 6.456s
You're referring to the famous number 42! It's a special number in the context of Douglas Adams' science fiction series "The Hitchhiker's Guide to the Galaxy." In the book, a supercomputer named Deep Thought is asked to find the "Answer to the Ultimate Question of Life, the Universe, and Everything." After 7.5 million years of calculations, Deep Thought reveals that the answer is... 42!

However, the characters in the book realize that the answer isn't as simple as they thought. They eventually discover that the answer is actually a joke, and that the "Ultimate Question" is actually unknown.

So, why is 42 special? Well, it's become a kind of cultural phenomenon, symbolizing the absurdity and complexity of the universe. It's also been interpreted as a nod to the idea that the answer to life's mysteries may not be straightforward or even definitive.

In a way, 42 has become a kind 

In [6]:
# Run these to remove the model from GPU VRAM.

del pipe
gc.collect()
torch.cuda.empty_cache()

Loading the model using device = "cuda:0" - loading the entire model on single GPU

In [7]:
# from transformers import pipeline
pipe = pipeline(
    "text-generation",
    model=model_id,
    dtype=torch.float16,
    device_map="cuda:0",
)

outputs = pipe(
    messages, max_new_tokens=200
)
print(outputs[0]["generated_text"][-1]['content'])

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 146/146 [00:01<00:00, 86.37it/s] 
[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You're referring to the famous "42" that appears in Douglas Adams' science fiction series "The Hitchhiker's Guide to the Galaxy." In the book, the number 42 is often cited as a special number, and its significance is a topic of much debate and speculation.

According to Douglas Adams, the author of the book, the number 42 is actually the "Answer to the Ultimate Question of Life, the Universe, and Everything." However, Adams never revealed the answer, and the book ends with the phrase "The answer, of course, is 42."

As for why 42 is considered a special number, there are a few theories:

1. **Mathematical significance**: Some mathematicians have calculated that the number 42 is actually the "Fibonacci sequence" multiplied by itself, which would make it a fundamental constant in mathematics.
2. **Philosophical significance**: The number 42 has become a symbol of the search for meaning and the quest for answers


Verifying the model loads on single GPU with nvidia-smi

In [8]:
!nvidia-smi

Wed Sep  9 19:03:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             41W /   70W |    2535MiB /  15360MiB |     19%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Timing analysis:

In [9]:
times = []

outputs = pipe(
        messages, max_new_tokens=200
    )

for _ in range(N):
    start = time.perf_counter()
    outputs = pipe(
        messages, max_new_tokens=200
    )
    time_taken = time.perf_counter() - start
    times.append(time_taken)
    print(f"Run {len(times)}: Time taken - {time_taken:.3f} sec")

avg_single = sum(times) / len(times)
print(f"\nAverage generation time over {N} runs: {avg_single:.3f}s")

print(outputs[0]["generated_text"][-1]['content'])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 1: Time taken - 4.723 sec


[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 2: Time taken - 4.302 sec


[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Run 7: Time taken - 4.226 sec


[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 8: Time taken - 4.278 sec


[transformers] Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Run 9: Time taken - 4.298 sec
Run 10: Time taken - 4.345 sec

Average generation time over 10 runs: 4.353s
The number 42 is indeed special, and its uniqueness has been widely discussed and debated. While there's no single, definitive answer, here are some reasons why 42 is considered a special number:

1. **The Answer to the Ultimate Question of Life, the Universe, and Everything**: In Douglas Adams' science fiction series "The Hitchhiker's Guide to the Galaxy," the supercomputer Deep Thought is asked to calculate the "Answer to the Ultimate Question of Life, the Universe, and Everything." After 7.5 million years of calculation, Deep Thought reveals that the answer is 42. This has become a cultural reference point, symbolizing the search for answers and the complexity of the universe.
2. **Mathematical significance**: 42 is a prime number, which means it can only be divided evenly by 1 and itself (42). This property makes it a fascinating number in mathematics, as it has unique propert

In [10]:
print(f"Based on above observations, on avg, running the model on single GPU is {((avg_auto - avg_single)/avg_auto) *100:.2f}% faster")

Based on above observations, on avg, running the model on single GPU is 32.57% faster
